In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import json
import os

In [ ]:
df = pd.read_csv('../../data/processed/processed_data.csv')

### Check data types
Incorrect data types could indicate errors in data.

In [ ]:
df.info()

In [ ]:
df.describe()

### Check for missing data
If there are missing values, fill or remove during data preparation. 

In [ ]:
df.isna().sum()

### Check for duplicates
If there are duplicates, remove or manage during data preparation.

In [ ]:
df.duplicated().any()

In [ ]:
df[df.duplicated()]

### Separate Values 
Create a separate list of values of whether things are categorical, continual numeric, or binary numeric

In [ ]:
cat_cols, cont_cols, binary_cols = [], [], []

for col in df.columns:
    if df[col].dtype == "str":
        cat_cols.append(col)
    else:
        if df[col].nunique() > 2:
            cont_cols.append(col)
        elif col != "fraud":
            binary_cols.append(col)


print("Categorical Columns:", cat_cols)

print("Continous Columns:", cont_cols)
print("Binary Columns:", binary_cols)

### Inspect Unique Values in the Dataset


In [ ]:
cols_to_drop = []
for col in cat_cols:
   unum = df[col].nunique()

   print(f"Unique numbers of {col}s:", unum)

   if unum == 1:
      cols_to_drop.append(col)

print("\nDropping columns due to constant values:", cols_to_drop)

for col in cols_to_drop:
   cat_cols.remove(col)

df.drop(columns=cols_to_drop, inplace=True)


### Create Datasplits
As data is time sensitive extracting features across the entire set causes leakage

In [ ]:
split_step = df["step"].quantile(0.7)
val_step = df["step"].quantile(0.85)

train_df = df[df["step"] <= split_step]
val_df = df[(df["step"] > split_step) & (df["step"] <= val_step)]
test_df = df[df["step"] > val_step]

### Get Customer Data
Inspect the global customer values of the dataset


In [ ]:
global_mean = train_df["amount"].mean()
global_std = train_df["amount"].std()

with open("../../data/processed/global_stats.json", "w") as f:
    json.dump({"transaction_mean": global_mean, "transaction_std": global_std}, f)

global_z_score = (train_df["amount"] - global_mean) / global_std
train_df["global_z_score"] = global_z_score

iqr = global_z_score.quantile(0.75) - global_z_score.quantile(0.25)
bin_width = 2 * iqr / (len(global_z_score) ** (1/3))
num_bins = int((global_z_score.max() - global_z_score.min()) / bin_width)

fig, axes = plt.subplots(2, 1,figsize=(10, 6))
axes[0].hist(global_z_score, bins=num_bins, range=(global_z_score.min(), global_z_score.max()))
axes[0].set_title("Global Z-Score Distribution of Transaction Amounts")
axes[0].set_xlabel("Global Z-Score")
axes[0].set_ylabel("Frequency")

axes[1].hist(global_z_score, log=True, range=(global_z_score.min(), global_z_score.max()), bins=num_bins)
axes[1].set_title("Global Z-Score Distribution (Log Scale)")
axes[1].set_xlabel("Global Z-Score")
axes[1].set_ylabel("Frequency")


In [ ]:
cust_freq = train_df["customer"].value_counts()
train_df["transactions_per_customer"] = train_df["customer"].map(cust_freq)

# Freedman-Diaconis rule to determine the number of bins for the histogram
iqr = cust_freq.quantile(0.75) - cust_freq.quantile(0.25)
bin_width = 2 * iqr / (len(cust_freq) ** (1/3))
num_bins = int((cust_freq.max() - cust_freq.min()) / bin_width)

fig, axes = plt.subplots(2, 1, figsize=(30, 20))

_, bin_edges, _ = axes[0].hist(cust_freq, bins=num_bins, range=(0, cust_freq.max()))
axes[0].set_xticks(bin_edges)
axes[0].grid(True, linestyle='--', alpha=0.7)

axes[0].set_title("Customer Transaction Frequencies")

axes[0].set_ylabel("Number of Customers")
axes[0].set_xlabel("Number of Transactions per Customer")


_, bin_edges, _ = axes[1].hist(cust_freq, bins=num_bins, log=True, range=(0, cust_freq.max()))
axes[1].set_xticks(bin_edges)
axes[1].grid(True, linestyle='--', alpha=0.7)

axes[1].set_title("Log-Scale Distribution")

axes[1].set_ylabel("Log(Number of Customers)")
axes[1].set_xlabel("Number of Transactions per Customer")

plt.show()

In [ ]:
avg_amount_per_cust = train_df.groupby("customer")["amount"].mean()
std_amount_per_cust = train_df.groupby("customer")["amount"].std().fillna(0)

train_df["avg_amount_per_customer"] = train_df["customer"].map(avg_amount_per_cust)
train_df["std_amount_per_customer"] = train_df["customer"].map(std_amount_per_cust)

amount_ratio = train_df["amount"] / train_df["customer"].map(avg_amount_per_cust)
log_amount_ratio = np.log1p(amount_ratio)

train_df["amount_ratio"] = amount_ratio
train_df["log_amount_ratio"] = log_amount_ratio

amount_zscore = (train_df["amount"] - train_df["avg_amount_per_customer"]) / (train_df["std_amount_per_customer"] + 1e-6)
train_df["amount_zscore"] = amount_zscore

iqr = avg_amount_per_cust.quantile(0.75) - avg_amount_per_cust.quantile(0.25)
bin_width = 2 * iqr / (len(avg_amount_per_cust) ** (1/3))
num_bins = int((avg_amount_per_cust.max() - avg_amount_per_cust.min()) / bin_width)

fig, axes = plt.subplots(4, 1, figsize=(30, 20))

_, bin_edges, _ = axes[0].hist(avg_amount_per_cust, bins=num_bins)
axes[0].set_title("Average Transaction Amount per Customer")
axes[0].set_ylabel("Number of Customers")
axes[0].set_xlabel("Average Transaction Amount")

axes[1].hist(avg_amount_per_cust, bins=num_bins, log=True)
axes[1].grid(True, linestyle='--', alpha=0.7)
axes[1].set_title("Log Distribution")
axes[1].set_ylabel("Log(Number of Customers)")
axes[1].set_xlabel("Average Transaction Amount")

iqr_ratio = amount_ratio.quantile(0.75) - amount_ratio.quantile(0.25)
ratio_bin_width = 2 * iqr_ratio / (len(amount_ratio) ** (1/3))
ratio_num_bins = int((amount_ratio.max() - amount_ratio.min()) / ratio_bin_width)

axes[2].hist(amount_ratio, bins=ratio_num_bins)
axes[2].grid(True ,linestyle='--', alpha=0.7)
axes[2].set_title("Amount Ratios")
axes[2].set_ylabel("Number of Transactions")
axes[2].set_xlabel("Amount / Avg Amount per Customer")

iqr_ratio = log_amount_ratio.quantile(0.75) - log_amount_ratio.quantile(0.25)
ratio_bin_width = 2 * iqr_ratio / (len(log_amount_ratio) ** (1/3))
ratio_num_bins = int((log_amount_ratio.max() - log_amount_ratio.min()) / ratio_bin_width)

axes[3].hist(log_amount_ratio, bins=ratio_num_bins)
axes[3].grid(True, linestyle='--', alpha=0.7)
axes[3].set_title("Log Ratio of Amount to Customer Average")
axes[3].set_ylabel("Log(Number of Transactions)")
axes[3].set_xlabel("Log(Amount / Avg Amount per Customer)")


plt.show()

In [ ]:
prev_step = train_df.groupby("customer")["step"].shift(1)
train_df["prev_step"] = prev_step

time_diff = train_df["step"] - train_df["prev_step"]
train_df["time_since_last_transaction"] = time_diff

In [ ]:
train_df 

In [ ]:
merchant_freq = df["merchant"].value_counts()
merchant_fraud_rate = df.groupby("merchant")["fraud"].mean()

fig, axes = plt.subplots(2, 1, figsize=(20, 8))
axes[0].bar(merchant_freq.index.astype(str), merchant_freq.values)
    
axes[0].set_title("Transaction Count per Merchant")
axes[0].set_xlabel("Merchant")
axes[0].set_ylabel("Number of Transactions")
axes[0].set_xticklabels(merchant_freq.index.astype(str), rotation=90)
axes[0].grid(axis="y", linestyle="--", alpha=0.7)

plt.show()